# Power-SMC demo

An open reproduction of *Power-SMC: Low-Latency Sequence-Level Power Sampling for Training-Free LLM Reasoning* (arXiv:2602.10273).

This notebook runs top to bottom. The first half needs only a CPU and proves the sampler is correct. The second half needs a GPU (a free Colab or Kaggle T4 is enough) and reproduces the accuracy-vs-latency story on MATH500.

## 0. Setup

In [ ]:
# On Colab / Kaggle, clone the repo and install. Locally, skip this cell.
# !git clone https://github.com/KarthikSubramanian07/power-smc.git
# %cd power-smc
!pip install -q numpy scipy matplotlib
# For the GPU section below also: torch transformers datasets accelerate bitsandbytes

## 1. Correctness on a toy model (CPU)

The toy model has a closed-form joint distribution, so we can enumerate every sequence and compute the exact power distribution. We then confirm the SMC particle set converges to it as the number of particles grows.

In [ ]:
import numpy as np
from power_smc import ToyMarkovModel, enumerate_exact, power_smc, total_variation

alpha = 4.0
model = ToyMarkovModel(n_real=3, max_len=4, seed=1, spread=2.0)
exact = enumerate_exact(model, alpha)
print(f'{len(exact.sequences)} terminated sequences; base and power both sum to 1')

for N in [4, 16, 64, 256, 512]:
    est = np.zeros(len(exact.sequences))
    runs = 150
    for s in range(runs):
        r = power_smc(model, None, alpha, n_particles=N, kappa=0.5, max_tokens=10, seed=1000 + s)
        est += r.weighted_distribution(exact.sequences)
    est /= runs
    print(f'N={N:4d}   TV to exact power distribution = {total_variation(est, exact.power_probs):.4f}')

## 2. The optimal proposal minimizes weight variance (CPU)

Theorem 1 says the temperature-`1/alpha` proposal is the unique prefix-only proposal that minimizes the incremental-weight variance. We confirm it in closed form: the variance is exactly zero at `tau = 1/alpha` and positive everywhere else.

In [ ]:
from power_smc import TemperatureProposal, conditional_weight_variance

_, logp = model.prefill(None, 1)
row = logp[0]
for tau in [0.1, 0.2, 0.25, 0.3, 0.5, 1.0, 2.0]:
    v = conditional_weight_variance(row, TemperatureProposal(tau), alpha)
    mark = '  <- 1/alpha' if abs(tau - 1 / alpha) < 1e-9 else ''
    print(f'tau={tau:.2f}   Var[w|prefix]={v:.3e}{mark}')

You can also regenerate the committed plots with a single command:

In [ ]:
!python validate.py

## 3. A real model (GPU)

From here you need a GPU. We load a small reasoning model in 4-bit and sample from the power distribution with a handful of particles.

In [ ]:
from power_smc.hf_model import HFModel
from power_smc import power_smc

hf = HFModel('Qwen/Qwen2.5-1.5B-Instruct', device='cuda', load_in_4bit=True)
question = 'If 3x + 7 = 22, what is x? Put the final answer in \\boxed{}.'
prompt_ids = hf.encode_chat(question, system='Please reason step by step, and put your final answer within \\boxed{}.')
result = power_smc(hf, prompt_ids, alpha=4.0, n_particles=8, kappa=0.5, max_tokens=512, seed=0)
print(hf.decode_text(result.output))
print('resampled at steps:', result.resample_steps)

## 4. MATH500 accuracy and latency (GPU)

Run each method over a subset, then build the headline plot. Increase `--subset` for the final numbers.

In [ ]:
!python experiments.py run --method baseline  --model Qwen/Qwen2.5-1.5B-Instruct --subset 50 --out results/math500_baseline.csv
!python experiments.py run --method power-smc --model Qwen/Qwen2.5-1.5B-Instruct --subset 50 --particles 8 --alpha 4 --out results/math500_power_smc.csv
!python experiments.py plot --inputs results/math500_baseline.csv results/math500_power_smc.csv --out results/plots/accuracy_latency.png

For the MH reference, clone https://github.com/aakaran/reasoning-with-sampling and run its MATH500 power-sampling script, then add its CSV to the `plot` command above. See the README for details.